# Session 2: fit k(T), then design the experiment
**A. Real-world problem.** Each measurement of a rate constant costs a day of lab work. You can
afford six temperatures. Which six give the most precise activation energy?

**Plan.** (1) the fit as a function with units, (2) the standard error of Ea from the spread
of 1000/T, (3) the modified Arrhenius form k = A T^n exp(-Ea/(R T)), (4) a design rule
checked by simulation.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R_GAS = 8.314462618          # J/(mol K)
A_TRUE = 1.0e13              # 1/s
EA_TRUE = 150.0              # kJ/mol

## Step 1: model, data, fit
ln k = ln A - (Ea/R)(1/T). With x = 1000/T the slope is s = -Ea/(1000 R), so Ea[kJ/mol] = -s R.
If the scatter of ln k differs from point to point, pass `sigma_ln` (one number per point):
numpy.polyfit then weights each point by 1/sigma.

In [ ]:
def arrhenius_k(T, A=A_TRUE, Ea_kJ=EA_TRUE, n=0.0):
    """Modified Arrhenius k = A T^n exp(-Ea/(R T)); n = 0 is the plain law. Units of A: 1/s when n = 0."""
    T = np.asarray(T, dtype=float)
    return A * T ** n * np.exp(-Ea_kJ * 1000.0 / (R_GAS * T))


def synthetic(T, rel_noise=0.05, seed=0, A=A_TRUE, Ea_kJ=EA_TRUE, n=0.0):
    """Measured k = true k times exp(rel_noise * N(0,1)): multiplicative (lognormal) error."""
    rng = np.random.default_rng(seed)
    T = np.asarray(T, dtype=float)
    return arrhenius_k(T, A, Ea_kJ, n) * np.exp(rel_noise * rng.standard_normal(T.size))


def fit_arrhenius(T, k, sigma_ln=None):
    """Fit ln k = ln A - (Ea/R)/T. Returns (A, Ea in kJ/mol)."""
    T, k = np.asarray(T, dtype=float), np.asarray(k, dtype=float)
    w = None if sigma_ln is None else 1.0 / np.asarray(sigma_ln, dtype=float)
    s, b = np.polyfit(1000.0 / T, np.log(k), 1, w=w)
    return float(np.exp(b)), float(-s * R_GAS)

## Step 2: how precise is Ea?
For equal-size errors sigma in ln k, the standard error of the slope is sigma / sqrt(Sxx)
with Sxx = sum (x - xbar)^2 and x = 1000/T. In kJ/mol: SE(Ea) = R * sigma / sqrt(Sxx).
A wide spread of x (a big Sxx) means a small error.

In [ ]:
def sxx(T):
    x = 1000.0 / np.asarray(T, dtype=float)
    return float(np.sum((x - x.mean()) ** 2))


def ea_se(T, sigma_ln):
    """Standard error of Ea in kJ/mol for temperatures T and error sigma_ln (no unit) in ln k."""
    return float(R_GAS * sigma_ln / np.sqrt(sxx(T)))


def even_in_inverse_T(Tmin, Tmax, n):
    """n temperatures evenly spaced in 1/T between Tmin and Tmax (K)."""
    return 1.0 / np.linspace(1.0 / Tmax, 1.0 / Tmin, n)


def simulated_ea_std(T, sigma_ln, repeats=4000, seed=3):
    """Spread (standard deviation) of the fitted Ea over many synthetic experiments."""
    rng = np.random.default_rng(seed)
    T = np.asarray(T, dtype=float)
    x = 1000.0 / T
    eas = np.empty(repeats)
    for i in range(repeats):
        y = np.log(arrhenius_k(T)) + sigma_ln * rng.standard_normal(T.size)
        eas[i] = -np.polyfit(x, y, 1)[0] * R_GAS
    return float(eas.std(ddof=1)), float(eas.mean())

## Step 3: the modified Arrhenius form
Many fits use k = A T^n exp(-Ea/(R T)), so ln k = ln A + n ln T - (Ea/R)(1/T). That is a
plane fit with three unknowns (ln A, n, Ea), solved by linear least squares.

In [ ]:
def fit_modified(T, k):
    """Returns (A, n, Ea in kJ/mol) from linear least squares on [1, ln T, -1000/T]."""
    T, k = np.asarray(T, dtype=float), np.asarray(k, dtype=float)
    M = np.column_stack([np.ones(T.size), np.log(T), -1000.0 / T])
    c = np.linalg.lstsq(M, np.log(k), rcond=None)[0]
    return float(np.exp(c[0])), float(c[1]), float(c[2] * R_GAS)

## Step 4: see it
Left: the same six noisy points fitted with two different temperature windows. Right: the
standard error of Ea against the width of the 1000/T range, theory against simulation.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    sig = 0.05
    wide = even_in_inverse_T(500, 1500, 6)
    narrow = even_in_inverse_T(850, 1000, 6)
    print("wide   : SE(Ea) =", round(ea_se(wide, sig), 2), "kJ/mol, simulated", round(simulated_ea_std(wide, sig)[0], 2))
    print("narrow : SE(Ea) =", round(ea_se(narrow, sig), 2), "kJ/mol, simulated", round(simulated_ea_std(narrow, sig)[0], 2))

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
    for T, name, col in ((wide, "500-1500 K", "tab:blue"), (narrow, "850-1000 K", "tab:red")):
        k = synthetic(T, sig, seed=5)
        A, Ea = fit_arrhenius(T, k)
        xg = np.linspace(0.6, 2.1, 20)
        ax[0].plot(1000 / T, np.log(k), "o", color=col, label=f"{name}: Ea = {Ea:.0f}")
        ax[0].plot(xg, np.log(A) - Ea / R_GAS * xg, color=col, lw=1)
    ax[0].set_xlabel("1000 / T (1/K x 1000)")
    ax[0].set_ylabel("ln k (k in 1/s)")
    ax[0].set_title("Same noise, two temperature windows")
    ax[0].grid(alpha=0.3)
    ax[0].legend()
    Tmaxs = np.linspace(900, 2000, 12)
    th = [ea_se(even_in_inverse_T(800, t, 6), sig) for t in Tmaxs]
    sm = [simulated_ea_std(even_in_inverse_T(800, t, 6), sig, repeats=1500)[0] for t in Tmaxs]
    ax[1].plot(Tmaxs, th, label="theory R sigma / sqrt(Sxx)")
    ax[1].plot(Tmaxs, sm, "o", label="simulation")
    ax[1].set_xlabel("highest temperature Tmax (K), lowest fixed at 800 K")
    ax[1].set_ylabel("standard error of Ea (kJ/mol)")
    ax[1].set_title("A wider 1/T range pins Ea down")
    ax[1].grid(alpha=0.3)
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(OUT / "design_range.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `ea_se` to find the standard error for 6 points evenly spaced in 1000/T from 700 to 1000 K at sigma = 0.05, then from 500 to 1000 K, and report the ratio.
2. **Stretch.** Fit the modified form to `synthetic(T, 0.0, 0, 1e8, 100, 1.5)` data at 12 temperatures from 500 to 1500 K and confirm that you recover n = 1.5, then add 5 percent noise and see how far n moves.
3. **Explain.** Explain why putting all measurements at two extreme temperatures gives the smallest error in Ea but is a poor design in practice.

## Assessment
- **A. Conceptual.** Why does a wider range of 1000/T give a smaller error in Ea, even though the number of data points is the same?
- **B. Numerical.** For temperatures 600 to 1000 K in steps of 100 K and an error of 10 percent, what is the standard error of Ea?
- **C. Graph and simulation interpretation.** In the figure, the narrow window (850 to 1000 K) and the wide window (500 to 1500 K) each use six points. What standard errors does the theory give and what is their ratio?
- **D. Design and optimisation challenge.** Design: six measurements between 500 and 1500 K at 5 percent error. Compare six points evenly spaced in 1000/T with three at each end. Which is better for the error of Ea, and what do you lose?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    T = np.array([600.0, 700.0, 800.0, 900.0, 1000.0])
    checks.append(("worked example: Sxx of 1000/T for 600..1000 K step 100 is 0.2788", abs(L.sxx(T) - 0.2788) < 0.0001))
    checks.append(("worked example: SE(Ea) = 8.3145 * 0.05 / sqrt(0.2788) = 0.787 kJ/mol", abs(L.ea_se(T, 0.05) - 0.787) < 0.001))
    A, Ea = L.fit_arrhenius(T, L.synthetic(T, 0.05, seed=1))
    checks.append(("worked example: seed 1 gives Ea = 149.70 kJ/mol and A = 9.65e12 per s", abs(Ea - 149.70) < 0.01 and abs(A / 9.65e12 - 1.0) < 0.002))
    checks.append(("that Ea is within 2 standard errors of the true 150", abs(Ea - 150.0) < 2 * L.ea_se(T, 0.05)))
    A0, E0 = L.fit_arrhenius([500.0, 1000.0], L.arrhenius_k([500.0, 1000.0]))
    checks.append(("noise-free two-point fit returns 150 kJ/mol and 1e13", abs(E0 - 150.0) < 1e-6 and abs(A0 / 1e13 - 1.0) < 1e-9))
    A1, E1 = L.fit_arrhenius(T, L.synthetic(T, 0.05, seed=1), sigma_ln=np.full(5, 0.05))
    checks.append(("equal weights change nothing", abs(E1 - Ea) < 1e-9 and abs(A1 / A - 1.0) < 1e-9))
    wide = L.even_in_inverse_T(500, 1500, 6)
    narrow = L.even_in_inverse_T(850, 1000, 6)
    checks.append(("even_in_inverse_T includes both end temperatures", abs(wide[0] - 1500) < 1e-9 and abs(wide[-1] - 500) < 1e-9))
    checks.append(("wide window (500-1500 K): Sxx 1.244 and SE(Ea) 0.37 kJ/mol", abs(L.sxx(wide) - 1.2444) < 1e-3 and abs(L.ea_se(wide, 0.05) - 0.373) < 0.001))
    checks.append(("narrow window (850-1000 K): Sxx 0.0218 and SE(Ea) 2.82 kJ/mol", abs(L.sxx(narrow) - 0.0218) < 1e-4 and abs(L.ea_se(narrow, 0.05) - 2.82) < 0.01))
    checks.append(("the narrow window is 7.6 times worse than the wide one", abs(L.ea_se(narrow, 0.05) / L.ea_se(wide, 0.05) - 7.55) < 0.05))
    sd, mean = L.simulated_ea_std(wide, 0.05)
    checks.append(("simulation: spread of Ea (0.37) matches theory within 5 percent, mean unbiased", abs(sd / L.ea_se(wide, 0.05) - 1.0) < 0.05 and abs(mean - 150.0) < 0.02))
    sd2, _ = L.simulated_ea_std(narrow, 0.05)
    checks.append(("simulation: narrow-window spread matches theory within 5 percent", abs(sd2 / L.ea_se(narrow, 0.05) - 1.0) < 0.05))
    ends = np.array([500.0, 500.0, 500.0, 1500.0, 1500.0, 1500.0])
    checks.append(("design: three points at each end give SE 0.255, 32 percent below the even design", abs(L.ea_se(ends, 0.05) - 0.2546) < 0.001 and abs(1 - L.ea_se(ends, 0.05) / L.ea_se(wide, 0.05) - 0.317) < 0.005))
    T2 = np.linspace(500.0, 1500.0, 12)
    A2, n2, E2 = L.fit_modified(T2, L.arrhenius_k(T2, 1e8, 100.0, 1.5))
    checks.append(("modified Arrhenius recovers A = 1e8, n = 1.5, Ea = 100 kJ/mol", abs(A2 / 1e8 - 1) < 1e-6 and abs(n2 - 1.5) < 1e-8 and abs(E2 - 100.0) < 1e-6))
    checks.append(("n = 0 reduces the modified form to the plain law", abs(float(L.arrhenius_k(800.0, 1e13, 150.0, 0.0)) / 1607.67 - 1) < 1e-4))
    xs = 1000.0 / T
    checks.append(("hand calculation: mean of 1000/T is 1.2913", abs(xs.mean() - 1.2913) < 1e-4))
    checks.append(("doubling the error doubles SE(Ea): sigma 0.10 gives 1.57 kJ/mol", abs(L.ea_se(T, 0.10) - 1.575) < 0.001))
    Tm = np.linspace(500.0, 1500.0, 12)
    Mx = np.column_stack([np.ones(12), np.log(Tm), -1000.0 / Tm])
    Cv = np.linalg.inv(Mx.T @ Mx)
    checks.append(("modified form: ln A and n are almost perfectly anti-correlated (r = -0.9998)", abs(Cv[0, 1] / np.sqrt(Cv[0, 0] * Cv[1, 1]) + 0.9998) < 0.0001))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
